<a href="https://www.kaggle.com/code/criser2013/pytorch-b-sico?scriptVersionId=303233647" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Guía para principiantes de Pytorch

En esta guía presento los aspectos básicos de Pytorch: tensores, datasets, dataloader, definición del modelo, entrenamiento, prueba, guardado y carga del mismo. Esta guía está basada en la [guía de inicio rápido](https://docs.pytorch.org/tutorials/beginner/basics/intro.html) de Pytorch.

**Carga de librerías**

In [ ]:
import torch
import torch.optim.lr_scheduler as lr_scheduler
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

# Tensores
Son el objeto básico de los frameworks de machine learning, el concepto básico es que estos son una serie de arreglos numéricos anidado. Hay varias formas de crearlos como:
- Se pueden crear a partir de listas de Python como por ejemplo:
```
tensor = torch.tensor([[1,1],[2,2]])
```
- También si ya son arreglos de numpy `torch.from_numpy(np.array([[1,1],[2,2]]))`.
- O mediante valores constantes: `torch.zeros(<dims>)`, `torch.ones(<dims>)`, aleatorios `torch.rand(<dims>)` y conservando las propiedades de otros tensores: `torch.ones_like(<tensor>)`o `torch.rand_like(<tensor>)`.
- De forma predeterminada se guardan en CPU pero se pueden mover a la GPU. **Nota:** Si son muy grandes esto puede ser lento.
- Si son de una dimensión, se puede acceder al valor usando la función `tensor.item()`.
- Se accede a los elementos igual que en los arreglos de numpy.

## Atributos
- **Dimensiones:** `tensor.shape`.
- **Tipo de dato:** `tensor.dtype`.
- **Dispositivo:** `tensor.device`.

## Operaciones
Van de operaciones básicas como sumas, restas, multiplicación de matrices, etc. Una operación especial es mover un tensor de un dispositivo a otro, esto se logra con `tensor.to(<dispositivo>)`  
- **Convertir tensor a array de numpy:** `tensor.numpy()`.
- **Concatenando tensores:** Se permite concatenar varios tensores según un eje o dimensión con `torch.cat([tensor_1, tensor_2], dim=1)`

**Retorno de la operaciones:** Si se realiza una operación usando una función que termine con "_", la función devuelve `None` y el valor se aplica a la variable almacenada. Por ejemplo:

```
print(f"{tensor} \n")
tensor.add_(5)
print(tensor)
```

Suma 5 a cada elemento del tensor. Los funciones que no tengan esa nomenclatura, devolverán el resultado y no sobreescribirán el argumento.

**Ojo:** Si se convierte un arreglo de numpy a tensor o viceversa y se realiza un cambio en alguno de los dos, este provoca que ambos se actualicen dado que comparten espacio de memoria (paso por referencia).

## Broadcasting

Si desea realizar una operación elemento a elemento entre tensores de diferentes dimensiones, Pytorch tratará de redimensionar el tensor mas pequeño para que sea compatible con el más grande. Esto lo suele realizar rellenando las dimensiones faltantes con 1s, sin embargo, esto no se almacena en memoria.  
El proceso comienza añadiendo 1s desde la dimensión mas a la izquierda del tensor más pequeño replicando los datos (solo de forma conceptual)

```
Tensor A has shape (5, 3)

Tensor B has shape (1, 3)

The first dimension of Tensor B (with size 1) will be broadcasted to size 5.
```

Los requisitos para hacer broadcasting es que por cada dimensión en ambos tensores, el tamaño sea el mismo o una de las dimensiones del tensor más pequeño tenga tamaño 1. Esto no funcionará por ejemplo, en tesnores con dimensiones: `(2, 3)` y `(3, 2)`.

# Dataset y Dataloader

La clase `Dataset` almacena las instancias y etiquetas correspondientes a las instancias. Mientra que la clase `DataLoader` permite generar un iterable a partir del conjunto de datos para acceder fácilmente a las instancias por lotes.


## Datasets preexistentes
Pytorch incluye conjuntos de datos de distinto tipo: audio, visión computacional y texto, estos se pueden acceder desde la biblioteca especializada correspondiente como se muestra a continuación con `torchvision`:

```
from torchvision import datasets

datos = datasets.<Clase-dataset>(
    root: str,                # Ruta dónde se almacenarán los archivos
    train: bool ,             # True si se desea cargar la partición de entrenamiento. Para cargar la de pruebas use False.
    download: bool,           # Si desea descargar los datos si no están en la ruta especificada.
    trasnform,                # Tranformación a aplicar a las instancias (no se aplica a las etiquetas)
    transform_target,         # Transformacióna  aplicar a las etiquetas de cada instancia
)
```

Los cbjetos `Dataset` pueden ser indexados como lista de Python `dataset[<indice>]`, lo anterior devolverá tanto los valores de los atributos como la etiqueta de la instancia.

## Usando conjuntos de datos personalizados
Debe crearse una clase que herede de la clase `Dataset` implementando los métodos: `__init__`, `__len__` y `__getitem__`.

**Ejemplo:**

```
import os
import pandas as pd
from torchvision.io import decode_image

class CustomImageDataset(Dataset):
    def __init__(self, annotations_file, img_dir, transform=None, target_transform=None):
        self.img_labels = pd.read_csv(annotations_file)
        self.img_dir = img_dir
        self.transform = transform
        self.target_transform = target_transform

    def __len__(self):
        return len(self.img_labels)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.img_labels.iloc[idx, 0])
        image = decode_image(img_path)
        label = self.img_labels.iloc[idx, 1]
        if self.transform:
            image = self.transform(image)
        if self.target_transform:
            label = self.target_transform(label)
        return image, label
```

- **Método `__init__`:** Es el constructor, se encarga de inicializar los datos y aplicar las transformaciones respectivas.
- **Método `__len__`:** Debe retornar la cantidad de instancias en el conjunto de datos.
- **Métdodo `__getitem__`:** Se encarga de retornar una instancia a partir de un indice. recibe como parámetro un índice (`idx`) y debe retornar una tupla de la forma `(<atributos>, <etiqueta>)`

In [ ]:
datos_entrenamiento = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    # Convierte la imagen de PIl o arreglo de numpy en un Tensor
    # Para hacer transformaciones personalizadas usar la clase Lambda que aplica una función a cada instancias
    transform=ToTensor()
)

datos_prueba = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor()
)

## Dataloader
Notese que el objeto `Dataset` devuelve una instancia a la vez. Para aprovechar las capacidades de multiprocesamiento se usa el `DataLoader`. `Dataloader` retorna un lote de instancias (un tensor de dimensión `[batch_size, <num-atribs>, <dim-etiquetas>]`. Para acceder a los elementos debe convertise el objeto en la clase `iterable` y recorrerse usando la función `next`.

```
# Display image and label.
train_features, train_labels = next(iter(train_dataloader))
print(f"Feature batch shape: {train_features.size()}")
print(f"Labels batch shape: {train_labels.size()}")
img = train_features[0].squeeze()
label = train_labels[0]
plt.imshow(img, cmap="gray")
plt.show()
print(f"Label: {label}")
```

In [ ]:
BATCH_SIZE = 64

dataloader_entrenamiento = DataLoader(datos_entrenamiento, batch_size=BATCH_SIZE)
dataloader_prueba = DataLoader(datos_prueba, batch_size=BATCH_SIZE)

for x,y in dataloader_prueba:
    print(f"Dimensiones de x [Muestras, Canales, Alto, Ancho]: {x.shape}")
    print(f"Dimensiones de y: {y.shape}")
    break

# Modelos

Para definir un modelo, este debe heredar de la clase `torch.nn.Module` e inicializar las capas de la red en el constructor (método `__init__`). El paso hacia delante de la red se implementa en el método `forward` de la red. Este método solo debe retornar la salida de aplicar las capas, por ejemplo, en problemas de clasificación debe retornar los logits y no el valor que arroje la función de activación (sigmoide o softmax).

Al igual que con los tensores, un modelo puede ser movido a la GPU usando `<clase-modelo>.to(<dispositivo>)`. Para ejecuar el modelo solo debe hacerse `<clase-modelo>(<datos>)` y este llamará de forma predeterminada al método `forward` de la clase.

In [ ]:
# Verificando si hay una GPU disponible
# De forma predeterminada TODO se mantiene en el CPU
dispositivo = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class RedNeuronal(nn.Module):
    def __init__(self):
        super(RedNeuronal, self).__init__()
        # Capa para aplanar la entrada bidimensional
        self.aplanamiento = nn.Flatten()
        # El objeto "Sequential" ejecutará secuencialmente cada una de las operaciones que especifiquemos
        # en el orden que se establezca. También se puede definir cada capa u operación como un atributo
        # pero deberemos llamar nosotros mismos a cada uno dentro del método "forward".
        self.capas = nn.Sequential(
            # Esta es una capa lineal (de la forma Wx+b). Recibe como entrada 784 atributos y consta de 512 neuronas/salidas
            nn.Linear(28*28, 512),
            # Ejecuta la función relu sobre cada elemento de la salida en la capa anterior
            nn.ReLU(),
            nn.Linear(512,512),
            nn.ReLU(),
            nn.Linear(512,10)
        )

    def forward(self, x):
        x = self.aplanamiento(x)
        logits = self.capas(x)
        return logits


modelo = RedNeuronal().to(dispositivo)

**Definiendo el optimizador y la función de error**  
Existe una característica denominada "learning rate scheduler", este permite ajustar la tasa de aprendizaje dinámicamente para que se adapte al proceso de entrenamiento. Lo anterior se logra gracias a la declaración de la constante `SCHEDULER`, dónde establecemos que la tasa de aprendizaje se reducirá 0.05 cada 10 iteraciones.

In [ ]:
FUNCION_PERDIDA = nn.CrossEntropyLoss()
OPTIMIZADOR = torch.optim.SGD(modelo.parameters(), lr= 1e-3)
SCHEDULER = lr_scheduler.StepLR(OPTIMIZADOR, step_size=10, gamma=0.05)

# Entrenamiento y prueba del modelo

Para entrenar el modelo, se utiliza el algoritmo de retropropagación, esto se logra con el método `backward` de la función de error que se haya escogido. De forma predeterminada, los tensores almacnan un historial de los gradientes al colocar el atributo `requires_grad=True`. El calculo de las derivadas y la actualización de los parámetros de la red se realizan automaticamente.

El entrenamiento de los modelos se realiza de la siguiente forma:
 1. Se debe recorrer cada lote del conjunto de datos trayendolos mediante el objeto `DataLoader`.
 2. Hacemos el paso hacia adelante usando las instancias de cada lote al modelo.
 3. Calculamos la función de error a partir de la salida del modelo y la salida esperada para cada instancia.
 4. Se calculan los gradientes de la función de error a partir de los parámetros del modelo (con la instrucción `<funcion-perdida>.backward()`.
 5. Se actualizan los parámetros a partir de los gradientes (mediante la instrucción `<modelo>.step()`).
 6. Reiniciamos los gradientes de los parámetros a 0 (con la instrucción `<optimizador>.zero_grad()`).
 7. Si se utiliza "learning rate scheduling", debe hacerce el llamado a `<scheduler>.step()` luego de hacer el mismo llamado al optimizador para que ajuste la tasa de aprendizaje (solo la adaptará cuando se cumpla el número de iteraciones que establecimos).

**Función de entrenamiento**

In [ ]:
def entrenar(dataloader, modelo, funcion_perdida, optimizador):
    TAM = len(dataloader.dataset)
    # Establecemos que el modelo está en modo de entrenamiento, por lo que se activan las capas de 
    # dropout y normalización
    modelo.train()
    for lote, (X, y) in enumerate(dataloader):
        X, y = X.to(dispositivo), y.to(dispositivo)

        # Calculo del error
        y_gorro = modelo(X)
        # No es necesario usar la función softmax ya que la
        # función de error la calcula sobre la entrada
        error = funcion_perdida(y_gorro, y)

        # Retropropagación
        # Calculo de los gradientes
        error.backward()
        # Actualización de los parámetros
        optimizador.step()
        # Reiniciamos los gradientes a 0 para que no hayan problemas en la próxima actualización
        optimizador.zero_grad()

        if lote % 100 == 0:
            perdida, instancias = error.item(), (lote + 1) * len(X)
            print(f"Error: {perdida:>7f}  [{instancias:>5d}/{TAM:>5d}]")
    SCHEDULER.step()

En ocasiones, querremos que el modelo  no almacene el historial de los gradientes, siendo el caso más común cuando solo requerimos que el modelo haga el paso haciada adelante. Para esto, debemos crear un contexto llamando a la función `torch.no_grad()`. Usualmente esto se realiza cuando ya hemos entrenado el modelo y queremos probarlo con algunas instancias.

**Función de prueba**

In [ ]:
def probar(dataloader, modelo, funcion_perdida):
    TAM = len(dataloader.dataset)
    NUM_LOTES = len(dataloader)
    # Coloca el modelo en modo de inferencia, por lo que desactiva capas como dropout y normalización
    modelo.eval()

    error, aciertos = 0,0
    # Desactivamos el guardado automático de los gradientes.
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(dispositivo), y.to(dispositivo)
            y_gorro = modelo(X)
            aciertos += (y_gorro.argmax(1) == y).type(torch.float).sum().item()
            error += funcion_perdida(y_gorro, y).item()
    error /= NUM_LOTES
    aciertos /= TAM
    print(f"\nError promedio: {error:>8f}, accuracy {(100*aciertos):0.2f}%")

Finalmente, ya que se ha definido el modelo, función de pérdida, optimizador y las funciones de entrenamiento y prueba, solo resta entrenar el modelo. Para esto se implementa un bucle de `n` iteraciones que optimizará los parámetros del modelo recorriendo `n` veces el conjunto de entrenamiento.

**Bucle de entrenamiento del modelo**

In [ ]:
ITERS = 25

for i in range(ITERS):
    print(f"Iteración {i+1}\n---------------------------------------")
    entrenar(dataloader_entrenamiento, modelo, FUNCION_PERDIDA, OPTIMIZADOR)
    probar(dataloader_prueba, modelo, FUNCION_PERDIDA)
    print("---------------------------------------")

print("¡Entrenamiento terminado!")

# Guardado del modelo

Los modelos de Pytorch almacenan el estado de sus parámetros en un diccionario. Se puede obtener este diccionario con el método `<modelo>.state_dict()`.  
Para guardar el estado del modelo se llama a la función `torch.save(<diccionario-estado>, <ruta-archivo>)`.

In [ ]:
torch.save(modelo.state_dict(), "modelo.pt")
print("Estado del modelo guardado en el archivo 'modelo.pt'")

# Carga del modelo
1. Se crea una instancia de la clase que define al modelo.
2. Cargamos el estado del modelo usando la función `torch.load(<ruta-archivo>, weights_only=True)`.
3. Asignamos ese estado al modelo con el método `<modelo>.load_state_dict(<diccionario-estado>)`.

In [ ]:
modelo = RedNeuronal().to(dispositivo)
modelo.load_state_dict(torch.load("modelo.pt", weights_only=True))

# Realizando la inferencia de una instancia individual

In [ ]:
classes = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat", "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

modelo.eval()

x, y = datos_prueba[0][0], datos_prueba[0][1]
with torch.no_grad():
    x = x.to(dispositivo)
    y_gorro = modelo(x)
    y_pred, y = classes[y_gorro[0].argmax(0)], classes[y]
    print(f'Valor predicho: "{y_pred}", valor real: "{y}"')